## Import

In [1]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import csv
from bertopic.vectorizers import ClassTfidfTransformer
from evaluation import evaluate_model

df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
classes = list(df["gen"])

c:\Users\aleblu\AppData\Local\miniconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:12<00:00,  2.57it/s]


In [6]:
#from sklearn.cluster import KMeans

run_name = ""

# Hyperparameters

n_neighbors = 30 # 15 -> BEST 30
n_components = 5 # 5 -> BEST 5
random_state = [0, 1, 37, 42, 73]
min_dist = 0.1
min_cluster_size = 10 # 10 -> BEST 10
min_df = 2 # 2 -> BEST 2 BUT 1 GOOD
ngram_range = (1, 2) # (1, 2) -> BEST (1, 2)
top_n_words = 10 # 10 -> BEST 10

# Data saving

data = []

# Setup different models

cluster_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=min_df, ngram_range=ngram_range)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # enable by default --> BEST DEFAULT

# Train

for seed in random_state:
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=seed)

    topic_model = BERTopic(

        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        #ctfidf_model=ctfidf_model,

        # Hyperparameters
        top_n_words=top_n_words,
        n_gram_range=ngram_range,
        min_topic_size="auto", #use HDBSCAN
        verbose=True,

        # General parameters
        calculate_probabilities=True,
        language="english"
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)

    # Evaluate model

    scores = evaluate_model(topic_model, docs, topics, embeddings, topk=top_n_words)  
    data.append([seed, scores[0], scores[1], scores[2], scores[3]])

df = pd.DataFrame(data, columns=["seed", "c_v", "c_npmi", "t_D", "S"])
df

2025-02-04 15:52:45,416 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-04 15:52:48,130 - BERTopic - Dimensionality - Completed ✓
2025-02-04 15:52:48,131 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-04 15:52:48,177 - BERTopic - Cluster - Completed ✓
2025-02-04 15:52:48,180 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-04 15:52:48,245 - BERTopic - Representation - Completed ✓
2025-02-04 15:53:04,460 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-04 15:53:07,323 - BERTopic - Dimensionality - Completed ✓
2025-02-04 15:53:07,324 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-04 15:53:07,383 - BERTopic - Cluster - Completed ✓
2025-02-04 15:53:07,387 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-04 15:53:07,465 - BERTopic - Representation - Completed ✓
2025-02-04 1

,seed,c_v,c_npmi,t_D,S
0,0,0.617864,-0.027490,0.562500,0.522039
1,1,0.597185,-0.040008,0.582353,0.526477
2,37,0.588563,-0.056120,0.547059,0.528802
3,42,0.589325,-0.045290,0.582353,0.533737
4,73,0.538887,-0.085513,0.565000,0.561584


## TEST

Save model

In [4]:
embedding_model = "all-MiniLM-L6-v2"
topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

Save data

In [5]:
param_str = str({
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "min_dist": min_dist,
    "random_state": random_state,
    "min_cluster_size": min_cluster_size,
    "min_df": min_df,
    "ngram_range": ngram_range,
    "top_n_words": top_n_words
})


data_run = [[run_name, param_str, df["Diversity"].mean(), df["Coherence"].mean()]]

df_run = pd.DataFrame(data_run, columns=["run_name", "params", "diversity", "coherence"])

df_run.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"), mode="a", header=False, index=False)

Show results

In [ ]:
df_results = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"))
df_results

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [ ]:
topic_model.visualize_documents(docs)

## DYNAMIC

In [ ]:
topics_over_time = topic_model.topics_over_time(docs, classes)

topic_model.visualize_topics_over_time(topics_over_time)

In [11]:
a = topic_model.probabilities_
b = topic_model.get_document_info(docs)

In [ ]:
from sklearn.metrics import silhouette_score
import numpy as np

# Generate `X` and `labels` only for non-outlier topics (as they are technically not clusters)
umap_embeddings = topic_model.umap_model.transform(embeddings)
indices = [index for index, topic in enumerate(topics) if topic != -1]
X = umap_embeddings[np.array(indices)]
labels = [topic for index, topic in enumerate(topics) if topic != -1]

# Calculate silhouette score
silhouette_score = silhouette_score(X, labels)
silhouette_score